In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/DARPA_OPTC_EDA')
REPO_URL = 'https://github.com/farzaandalwai/DARPA_OPTC_EDA.git'
BRANCH = 'eda08'
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / '.git').exists() or git('remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('STOP: unexpected repository at code path')
    if git('status', '--porcelain'):
        raise RuntimeError('STOP: preserve local code changes before syncing')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)


In [ ]:
import hashlib

RUN_COMMIT = '3636eb8fee627034063c3800d202dbbc08822546'
print('branch:', git('branch', '--show-current'))
print('git HEAD:', git('rev-parse', 'HEAD'))
if git('branch', '--show-current') != BRANCH:
    raise RuntimeError('STOP: wrong branch')
subprocess.run(['git', '-C', str(REPO), 'merge-base', '--is-ancestor', RUN_COMMIT, 'HEAD'], check=True)
for name in ('src/eda/build_period_heterogeneous_graph.py', 'src/eda/eda_05_entity_dictionary.py',
             'reports/sysclient0201_hybrid_graph_real_run_v1.json'):
    expected = subprocess.check_output(['git', '-C', str(REPO), 'show', f'{RUN_COMMIT}:{name}'])
    current = subprocess.check_output(['git', '-C', str(REPO), 'show', f'HEAD:{name}'])
    if current != expected or (REPO / name).read_bytes() != expected:
        raise RuntimeError(f'STOP: verified graph logic or real-run manifest changed: {name}')
print('Verified run ancestry and identical graph-builder/dependency/manifest bytes')


In [ ]:
import os

DATA_ROOT = Path('/content/drive/MyDrive/DARPA_OPTC_EDA')
if not os.path.ismount('/content/drive') or not DATA_ROOT.is_dir():
    raise RuntimeError('STOP: Google Drive is not mounted or project root is absent')
print('Google Drive mounted:', True)
for role in ('verified_benign', 'evaluation'):
    folder = DATA_ROOT / f'eda_10_sysclient0201_hybrid_graph_{role}_v1'
    print(role, folder, 'exists:', folder.is_dir())
    if not folder.is_dir():
        raise RuntimeError('STOP: expected published output folder missing')


# Guarded publication resume — no graph construction

Run all cells in order. Both periods are verified before any graph write. Existing mismatches stop; valid artifacts and historical publication markers remain untouched. The saved real-run audit is authoritative. The transport folder contains 17 lossless byte chunks of only the two missing training edge files, not new graph data.

Each chunk, combined bytes, original full SHA256, and row count are verified. Large outputs are written to unique temporary filenames in the destination folder, then atomically promoted without overwriting. If Drive does not support no-replace promotion, the runner stops and preserves the verified temporary file; it never falls back to an overwriting rename. No recovery bytes means INCOMPLETE, not a rebuild.


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)


In [ ]:
import uuid

PARTS_ROOT = DATA_ROOT / 'eda_10_sysclient0201_hybrid_graph_publication_transport_v1'
TRANSPORT_MANIFEST = REPO / 'reports/sysclient0201_hybrid_graph_resume_transport_v1.json'
# Optional: an already-completed output root under /content, never raw EDA10 inputs.
COMPLETED_ROOT = None
# Recovery parts are needed only for missing outputs, never for valid graphs.
REPORT_ROOT = PARTS_ROOT if PARTS_ROOT.is_dir() else DATA_ROOT
REPORT = REPORT_ROOT / f'publication_verification_{uuid.uuid4().hex}.json'
command = [sys.executable, '-m', 'src.eda.resume_sysclient0201_hybrid_graph_publication',
           '--repo', str(REPO), '--data-root', str(DATA_ROOT),
           '--parts-root', str(PARTS_ROOT), '--transport-manifest', str(TRANSPORT_MANIFEST),
           '--report', str(REPORT)]
if COMPLETED_ROOT is not None:
    command += ['--completed-root', str(COMPLETED_ROOT)]
result = subprocess.run(command, cwd=REPO)
print('Saved verification report:', REPORT)
if result.returncode:
    raise RuntimeError('INCOMPLETE — read the report; preserve mismatched/temporary files, no rebuild')
print('COMPLETE — both published graph folders match the completed real run. Stop here.')
